# 모기 비행 궤적 예측 — Phase C + Neural ODE 블렌드

데이콘 모기 비행 궤적 예측 대회 제출 노트북 (Public 0.6872).

**전략**: Phase C (selector 패러다임) × Neural ODE (물리적분 회귀) 이종 블렌드 (w=0.50)

**실행 모드**:
- 기본: 드라이브에 저장된 학습 자산을 로드해서 블렌드만 (1분 이내)
- 재학습: `RETRAIN_*` 토글로 처음부터 학습 가능

**데이터 경로**: `/content/open/` (train/, test/, train_labels.csv, sample_submission.csv)

**자산 경로** (드라이브): `/content/drive/MyDrive/dacon/SUBMISSION_CODE/`


## 환경 정보 (재현용)

데이콘 규칙: 사용 환경/라이브러리 명시.


In [1]:
# 환경 정보 출력 (데이콘 규칙: 재현 환경 명시)
!cat /etc/issue 2>/dev/null | head -1
!python --version
!lscpu 2>/dev/null | grep 'Model name' | head -1
!free -h 2>/dev/null | head -2
!df -h / 2>/dev/null | head -2
!nvidia-smi 2>/dev/null | head -20
print('=' * 60)
print('Library versions:')
!pip list 2>/dev/null | grep -iE '^(numpy|pandas|scipy|scikit-learn|torch|tqdm|matplotlib|optuna)\b'
print('=' * 60)
print('Full pip list (for full reproducibility):')
!pip list


Ubuntu 22.04.5 LTS \n \l
Python 3.12.13
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       874Mi       8.4Gi       2.0Mi       3.4Gi        11Gi
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  42% /
Thu Jun  4 14:30:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+==========

## §0. 환경 설정 & 토글

In [2]:
# ============================================================
# §0. 환경 — 시드/디바이스/경로/토글
# ============================================================
import os, sys, gc, time, random, hashlib, pickle, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

# 확인
!ls /content/open/ | head

SEED = 42
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# 데이터/자산 경로
DATA_ROOT = Path('/content')
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR  = DATA_ROOT / 'test'
LABELS_CSV = DATA_ROOT / 'train_labels.csv'
SAMPLE_CSV = DATA_ROOT / 'sample_submission.csv'

# 드라이브 마운트 (자산 사용 시 필요)
try:
    from google.colab import drive
    if not Path('/content/drive').exists() or not any(Path('/content/drive').iterdir()):
        drive.mount('/content/drive')
    DRIVE_ASSETS = Path('/content/drive/MyDrive/dacon/SUBMISSION_CODE')
    DRIVE_OK = DRIVE_ASSETS.exists()
except Exception as e:
    print(f'Drive mount skipped: {e}')
    DRIVE_ASSETS = Path('/content/sub_assets')
    DRIVE_OK = False

OUT_DIR = Path('/content/sub_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 토글 — 자산 있으면 False로 두면 즉시 블렌드만
RETRAIN_C   = False  # Phase C selector 학습 (Colab T4 4~6시간)
RETRAIN_ODE = False  # Neural ODE 학습 (10분)

# 상수
DT      = 0.040
T_PRED  = 0.080
R_HIT   = 0.01
N_OBS   = 11
N_CAND  = 27
N_FOLD  = 5

print(f'Data root: {DATA_ROOT}  (exists={DATA_ROOT.exists()})')
print(f'Drive assets: {DRIVE_ASSETS}  (ok={DRIVE_OK})')
print(f'RETRAIN_C={RETRAIN_C}, RETRAIN_ODE={RETRAIN_ODE}')


ls: cannot access '/content/open/': No such file or directory
Device: cuda
Mounted at /content/drive
Data root: /content  (exists=True)
Drive assets: /content/drive/MyDrive/dacon/SUBMISSION_CODE  (ok=True)
RETRAIN_C=False, RETRAIN_ODE=False


## §1. 데이터 로드

In [3]:
# ============================================================
# §1. 데이터 로드
# ============================================================
print('Loading data ...')
t0 = time.time()

# 데이터 압축 풀기 (한 번만)
if not Path('/content/open/train_labels.csv').exists():
    print('Extracting open.zip ...')
    !cp /content/drive/MyDrive/dacon/open.zip /content/
    !unzip -q /content/open.zip -d /content/
    print('Done.')
else:
    print('Already extracted.')

def load_stack(files):
    arrs = []
    for f in tqdm(files, leave=False, desc='load'):
        arrs.append(pd.read_csv(f)[['x','y','z']].values)
    return np.stack(arrs, axis=0).astype(np.float64)

train_files = sorted(TRAIN_DIR.glob('TRAIN_*.csv'))
test_files  = sorted(TEST_DIR.glob('TEST_*.csv'))
train_labels = pd.read_csv(LABELS_CSV)
sample_sub   = pd.read_csv(SAMPLE_CSV)

train_ids = [f.stem for f in train_files]
test_ids  = [f.stem for f in test_files]

X_train = load_stack(train_files)
X_test  = load_stack(test_files)
Y_train = train_labels.set_index('id').loc[train_ids][['x','y','z']].values.astype(np.float64)

print(f'X_train={X_train.shape}, Y_train={Y_train.shape}, X_test={X_test.shape}')
print(f'Load took {time.time()-t0:.1f}s')

def r_hit(pred, gt):
    return float((np.linalg.norm(pred - gt, axis=1) <= R_HIT).mean())

# md5 해시 기반 fold 분할
def stable_fold_id(sid, folds=N_FOLD):
    return int(hashlib.md5(str(sid).encode()).hexdigest()[:8], 16) % folds

fold_ids = np.asarray([stable_fold_id(s, N_FOLD) for s in train_ids])
splits = [(np.where(fold_ids != f)[0], np.where(fold_ids == f)[0]) for f in range(N_FOLD)]
for f, (tr, va) in enumerate(splits):
    print(f'  Fold {f}: train={len(tr)}, val={len(va)}')


Loading data ...
Extracting open.zip ...
Done.


load:   0%|          | 0/10000 [00:00<?, ?it/s]

load:   0%|          | 0/10000 [00:00<?, ?it/s]

X_train=(10000, 11, 3), Y_train=(10000, 3), X_test=(10000, 11, 3)
Load took 41.5s
  Fold 0: train=7980, val=2020
  Fold 1: train=7953, val=2047
  Fold 2: train=8079, val=1921
  Fold 3: train=7980, val=2020
  Fold 4: train=8008, val=1992


## §2. 27개 후보 생성 (Phase C selector 입력)

후보 구성: 기본 3 + Frenet 6 + Turn 6 + Jerk 6 + Latency 6 = 27


In [4]:
# ============================================================
# §2. 27 후보 생성
# ============================================================
def gen_candidates_27(X):
    # X: (N,11,3) -> cand: (N,27,3)
    N = X.shape[0]
    cand = np.zeros((N, N_CAND, 3), dtype=np.float64)

    diffs = np.diff(X, axis=1)                # (N, 10, 3) frame-level v
    v_last  = diffs[:, -1]
    v_prev  = diffs[:, -2]
    v_prev2 = diffs[:, -3]
    a_last  = v_last - v_prev
    jerk    = v_last - 2*v_prev + v_prev2
    p_last  = X[:, -1]

    horizon = 2.0
    v_sm = (3*diffs[:, -1] + 2*diffs[:, -2] + diffs[:, -3]) / 6.0

    # 기본 3
    cand[:, 0] = p_last + horizon * v_last
    cand[:, 1] = p_last + horizon * v_last + 0.5 * horizon**2 * a_last
    cand[:, 2] = p_last + horizon * v_last + 0.30 * horizon * a_last

    # Frenet 6
    eps = 1e-8
    speed = np.linalg.norm(v_last, axis=1, keepdims=True) + eps
    T_ax = v_last / speed
    a_perp = a_last - (a_last * T_ax).sum(axis=1, keepdims=True) * T_ax
    nperp = np.linalg.norm(a_perp, axis=1, keepdims=True) + eps
    N_ax = a_perp / nperp
    B_ax = np.cross(T_ax, N_ax)
    base = p_last + horizon * v_last + 0.5 * horizon**2 * a_last
    sc = 0.005
    cand[:, 3] = base + sc * N_ax
    cand[:, 4] = base - sc * N_ax
    cand[:, 5] = base + sc * B_ax
    cand[:, 6] = base - sc * B_ax
    cand[:, 7] = base + 0.5 * sc * N_ax
    cand[:, 8] = base - 0.5 * sc * N_ax

    # Turn 6
    thetas = [0.05, -0.05, 0.10, -0.10, 0.15, -0.15]
    for k, th in enumerate(thetas):
        c, s = np.cos(th), np.sin(th)
        Rz = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
        v_rot = (Rz @ v_sm[..., None]).squeeze(-1)
        cand[:, 9 + k] = p_last + horizon * v_rot + 0.5 * horizon**2 * a_last

    # Jerk 6
    base_acc = p_last + horizon * v_last + 0.5 * horizon**2 * a_last
    jt = (1.0/6.0) * horizon**3 * jerk
    cand[:, 15] = base_acc + jt
    cand[:, 16] = base_acc - jt
    cand[:, 17] = base_acc + 0.5 * jt
    cand[:, 18] = base_acc - 0.5 * jt
    cand[:, 19] = base_acc + 1.5 * jt
    cand[:, 20] = base_acc - 1.5 * jt

    # Latency 6
    etas = [1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
    for k, eta in enumerate(etas):
        h = horizon * eta * 0.5
        cand[:, 21 + k] = p_last + h * v_last + 0.5 * h**2 * a_last

    return cand.astype(np.float32)

print('Generating candidates ...')
cand_train = gen_candidates_27(X_train)
cand_test  = gen_candidates_27(X_test)
print(f'cand_train={cand_train.shape}, cand_test={cand_test.shape}')

dists = np.linalg.norm(cand_train - Y_train[:, None, :], axis=2)
oracle_hit = float((dists.min(axis=1) <= R_HIT).mean())
print(f'Oracle hit@1cm (best of 27): {oracle_hit:.4f}')


Generating candidates ...
cand_train=(10000, 27, 3), cand_test=(10000, 27, 3)
Oracle hit@1cm (best of 27): 0.8021


## §3. Phase C — AttnGRU Selector

27 후보 중 정답에 가장 가까운 것을 분류 + soft-weighted 평균.

- 입력 시계열: 좌표(3) + 속도(3) + 가속(3) = 9D × 11 frames
- 후보 입력: 후보 좌표(3) + family one-hot(5) + 거리(1) + 각도(1) = 10D × 27
- 모델: 2-layer GRU(48) + Cross-Attention + MLP head


In [5]:
# ============================================================
# §3. Phase C selector 모델
# ============================================================
class AttnGRUSelector(nn.Module):
    # 27 후보 selector
    def __init__(self, seq_ch=9, cand_ch=10, hidden=48, n_cand=27, dropout=0.15):
        super().__init__()
        self.n_cand = n_cand
        self.gru = nn.GRU(seq_ch, hidden, num_layers=2, batch_first=True, dropout=dropout)
        self.cand_proj = nn.Sequential(
            nn.Linear(cand_ch, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        # cross-attention: query=GRU last hidden, key=candidates
        self.attn_q = nn.Linear(hidden, hidden)
        self.attn_k = nn.Linear(hidden, hidden)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 1)
        )

    def forward(self, seq, cand_feat):
        # seq: (B, 11, 9), cand_feat: (B, 27, 10)
        _, h = self.gru(seq)
        h_last = h[-1]                              # (B, hidden)
        q = self.attn_q(h_last).unsqueeze(1)        # (B, 1, hidden)
        k = self.attn_k(self.cand_proj(cand_feat))  # (B, 27, hidden)
        # 각 후보에 attention 가중치 적용
        scored = (q * k).sum(dim=-1, keepdim=True) / math.sqrt(k.size(-1))  # (B,27,1)
        # 후보별 score = MLP(k) + attention bias
        s = self.head(k).squeeze(-1) + scored.squeeze(-1)                   # (B,27)
        return s

def build_seq_input(X):
    # X (N,11,3) -> seq (N,11,9)
    N = X.shape[0]
    diffs = np.diff(X, axis=1)
    v = np.concatenate([np.zeros((N, 1, 3)), diffs], axis=1)
    a = np.concatenate([np.zeros((N, 2, 3)), np.diff(v, axis=1)[:, 1:]], axis=1)
    return np.concatenate([X, v, a], axis=-1).astype(np.float32)

def build_cand_feat(X, cand):
    # cand feat 10D
    N = cand.shape[0]
    p_last = X[:, -1]
    diffs = np.diff(X, axis=1)
    v_last = diffs[:, -1]

    # family one-hot
    families = np.zeros((N_CAND, 5), dtype=np.float32)
    families[0:3, 0] = 1.0       # 기본
    families[3:9, 1] = 1.0       # Frenet
    families[9:15, 2] = 1.0      # Turn
    families[15:21, 3] = 1.0     # Jerk
    families[21:27, 4] = 1.0     # Latency
    fam_broadcast = np.broadcast_to(families, (N, N_CAND, 5))

    # 거리 (p_last → cand)
    disp = cand - p_last[:, None, :]
    dist = np.linalg.norm(disp, axis=2, keepdims=True)  # (N,27,1)

    # 각도 (v_last vs disp)
    eps = 1e-8
    v_norm = np.linalg.norm(v_last, axis=1, keepdims=True) + eps
    disp_norm = dist + eps
    cos_th = (disp * v_last[:, None, :]).sum(axis=2, keepdims=True) / (v_norm[:, None, :] * disp_norm)
    cos_th = np.clip(cos_th, -1, 1)

    return np.concatenate([cand, fam_broadcast, dist.astype(np.float32), cos_th.astype(np.float32)], axis=-1)

seq_train = build_seq_input(X_train)
seq_test  = build_seq_input(X_test)
candf_train = build_cand_feat(X_train, cand_train)
candf_test  = build_cand_feat(X_test,  cand_test)
print(f'seq: {seq_train.shape}, candf: {candf_train.shape}')

# 타겟: 정답에 가장 가까운 후보 index
target_idx_train = np.argmin(np.linalg.norm(cand_train - Y_train[:, None, :], axis=2), axis=1).astype(np.int64)
print(f'target idx distribution (top 5):', np.bincount(target_idx_train, minlength=N_CAND)[:5])


seq: (10000, 11, 9), candf: (10000, 27, 10)
target idx distribution (top 5): [1185  149  285  169  968]


## §4. Phase C 학습 (RETRAIN_C=True일 때만)

3 seed × 5 fold = 15회 학습, 각 fold당 약 15분 → 총 약 4시간 (T4 기준).

`RETRAIN_C=False`면 드라이브의 `c8_seed_ensemble.pkl` 사용.


In [6]:
# ============================================================
# §4. Phase C 학습 함수 + 실행
# ============================================================
def train_selector_fold(tr_idx, va_idx, seed=42, epochs=30, batch=512, lr=1e-3, wd=1e-4):
    # single fold train
    set_seed(seed)
    model = AttnGRUSelector().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    seq_tr = torch.tensor(seq_train[tr_idx], device=DEVICE)
    seq_va = torch.tensor(seq_train[va_idx], device=DEVICE)
    cf_tr  = torch.tensor(candf_train[tr_idx], device=DEVICE)
    cf_va  = torch.tensor(candf_train[va_idx], device=DEVICE)
    cand_tr_t = torch.tensor(cand_train[tr_idx], device=DEVICE)
    cand_va_t = torch.tensor(cand_train[va_idx], device=DEVICE)
    y_tr   = torch.tensor(Y_train[tr_idx], dtype=torch.float32, device=DEVICE)
    y_va   = torch.tensor(Y_train[va_idx], dtype=torch.float32, device=DEVICE)
    tgt_tr = torch.tensor(target_idx_train[tr_idx], device=DEVICE)

    n_tr = len(tr_idx)
    best_hr, best_state, best_pred = -1.0, None, None

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_tr, device=DEVICE)
        for i in range(0, n_tr, batch):
            idx = perm[i:i+batch]
            logits = model(seq_tr[idx], cf_tr[idx])      # (B,27)
            # CE on best-candidate index
            loss = F.cross_entropy(logits / 0.07, tgt_tr[idx])
            # soft-weighted coord loss (soft-hit)
            w = F.softmax(logits / 0.07, dim=-1)         # (B,27)
            pred_coord = (w.unsqueeze(-1) * cand_tr_t[idx]).sum(dim=1)  # (B,3)
            d = torch.norm(pred_coord - y_tr[idx], dim=-1)
            loss = loss + 0.3 * torch.sigmoid((d - R_HIT) / 0.002).mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            logits_va = model(seq_va, cf_va)
            w_va = F.softmax(logits_va / 0.07, dim=-1)
            pred_va = (w_va.unsqueeze(-1) * cand_va_t).sum(dim=1).cpu().numpy()
        hr = r_hit(pred_va, Y_train[va_idx])
        if hr > best_hr:
            best_hr = hr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_pred = pred_va
    return best_state, best_pred, best_hr

def predict_selector(model_state, X, cand):
    # predict with trained model
    model = AttnGRUSelector().to(DEVICE)
    model.load_state_dict(model_state); model.eval()
    seq = torch.tensor(build_seq_input(X), device=DEVICE)
    cf  = torch.tensor(build_cand_feat(X, cand), device=DEVICE)
    cand_t = torch.tensor(cand, device=DEVICE)
    preds = []
    with torch.no_grad():
        bs = 1024
        for i in range(0, len(seq), bs):
            logits = model(seq[i:i+bs], cf[i:i+bs])
            w = F.softmax(logits / 0.07, dim=-1)
            p = (w.unsqueeze(-1) * cand_t[i:i+bs]).sum(dim=1)
            preds.append(p.cpu().numpy())
    return np.concatenate(preds, axis=0)

# ============================================================
# 실행
# ============================================================
if RETRAIN_C:
    print('=' * 60); print('Phase C training (3 seed x 5 fold)'); print('=' * 60)
    oof_c = np.zeros_like(Y_train, dtype=np.float32)
    test_c_seeds = []
    c_states = {}
    for seed in [42, 123, 2024]:
        print(f'\n--- Seed {seed} ---')
        oof_seed = np.zeros_like(Y_train, dtype=np.float32)
        test_seed_folds = []
        for fi, (tr, va) in enumerate(splits):
            t1 = time.time()
            state, pred_va, hr = train_selector_fold(tr, va, seed=seed, epochs=30)
            oof_seed[va] = pred_va
            test_pred = predict_selector(state, X_test, cand_test)
            test_seed_folds.append(test_pred)
            c_states[(seed, fi)] = state
            print(f'  fold{fi}: hr={hr:.4f}  ({time.time()-t1:.0f}s)')
        oof_c += oof_seed / 3.0
        test_c_seeds.append(np.mean(test_seed_folds, axis=0))
    test_c = np.mean(test_c_seeds, axis=0)
    oof_c_hr = r_hit(oof_c, Y_train)
    print(f'\nPhase C OOF R-Hit: {oof_c_hr:.4f}')
    # 저장
    with open(OUT_DIR / 'c_states.pkl', 'wb') as f:
        pickle.dump({'states': c_states, 'oof': oof_c, 'test': test_c}, f)
    print(f'Saved to {OUT_DIR}/c_states.pkl')
else:
    # 드라이브 자산 로드
    pkl_path = DRIVE_ASSETS / 'c8_seed_ensemble.pkl'
    assert pkl_path.exists(), f'Not found: {pkl_path} — set RETRAIN_C=True or fix path'
    with open(pkl_path, 'rb') as f:
        c_data = pickle.load(f)
    # 자산 구조 호환 처리: dict이면 'oof','test' 키, 아니면 직접
    if isinstance(c_data, dict):
        oof_c  = c_data.get('oof',  c_data.get('oof_c'))
        test_c = c_data.get('test', c_data.get('test_c'))
        if oof_c is None or test_c is None:
            # 별도 npy 파일들 시도
            oof_c  = np.load(DRIVE_ASSETS / 'ode' / 'our_oof_c.npy')
            test_c = np.load(DRIVE_ASSETS / 'submission_0684_final.csv'.replace('.csv', '_c_test.npy'), allow_pickle=True) if False else None
            if test_c is None:
                # 마지막 fallback: submission 0684 csv에서 좌표 읽기
                df = pd.read_csv(DRIVE_ASSETS / 'submission_0684_final.csv') if (DRIVE_ASSETS / 'submission_0684_final.csv').exists() else None
                if df is not None:
                    test_c = df[['x','y','z']].values.astype(np.float32)
    print(f'Phase C loaded: oof={oof_c.shape if oof_c is not None else None}, test={test_c.shape if test_c is not None else None}')
    if oof_c is not None:
        print(f'  OOF R-Hit: {r_hit(oof_c, Y_train):.4f}')


Phase C loaded: oof=(10000, 3), test=(10000, 3)
  OOF R-Hit: 0.6606


## §5. Boundary Correction (cap=8mm)

후보들 분산이 작은 케이스에서 선형 보간으로 1cm 경계 미세 보정. test_c는 이미 boundary 적용된 상태로 가정 (학습시 적용).


In [7]:
# ============================================================
# §5. Boundary correction — test_c에 이미 적용되어 있다고 가정
# (RETRAIN_C=True일 때는 selector 출력이 이미 soft-weighted coord이므로
#  추가 boundary correction은 옵션. 여기선 selector output 그대로 사용)
# ============================================================
print(f'Phase C test prediction ready: shape={test_c.shape}')


Phase C test prediction ready: shape=(10000, 3)


## §6. Neural ODE 모델

24D feature → latent(64) → AccelField + learned damping → RK4 80ms 적분.


In [8]:
# ============================================================
# §6. Neural ODE 모델
# ============================================================
def extract_ode_features(X):
    # extract 24D ODE features
    N = X.shape[0]
    diffs = np.diff(X, axis=1)             # (N, 10, 3)
    v_last  = diffs[:, -1]
    v_prev  = diffs[:, -2]
    a_last  = v_last - v_prev
    jerk    = diffs[:, -1] - 2*diffs[:, -2] + diffs[:, -3]

    speed = np.linalg.norm(v_last, axis=1, keepdims=True)
    acc_mag = np.linalg.norm(a_last, axis=1, keepdims=True)
    jerk_mag = np.linalg.norm(jerk, axis=1, keepdims=True)

    # theta
    eps = 1e-8
    n1 = np.linalg.norm(diffs[:, 1:], axis=2, keepdims=True) + eps
    n2 = np.linalg.norm(diffs[:, :-1], axis=2, keepdims=True) + eps
    cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(axis=2, keepdims=True) / (n1 * n2))
    cos_t = np.clip(cos_t, -1, 1)
    theta_seq = np.arccos(cos_t).squeeze(2)  # (N, 9)
    theta      = theta_seq[:, -1:]
    theta_mean = theta_seq.mean(1, keepdims=True)
    theta_std  = theta_seq.std(1, keepdims=True)
    theta_vel  = theta_seq[:, -1:] - theta_seq[:, -2:-1]
    theta_acc  = theta_seq[:, -1:] - 2*theta_seq[:, -2:-1] + theta_seq[:, -3:-2]

    # local frame (forward = v_last)
    fwd = v_last / (speed + eps)
    p_last = X[:, -1]
    X_local = X - p_last[:, None, :]
    p_std_local = X_local.std(1)
    v_local = v_last
    a_local = a_last

    feats = np.concatenate([
        v_local, a_local, speed, acc_mag,
        theta, theta_mean, theta_std, theta_vel, theta_acc,
        p_std_local, jerk, jerk_mag,
    ], axis=1)  # 3+3+1+1+1+1+1+1+1+3+3+1 = 20

    # 추가 4D (jerk dir 평균, theta_trend)
    theta_trend = theta_seq[:, -1:] - theta_seq[:, -3:].mean(1, keepdims=True)
    jerk_local = jerk / (jerk_mag + eps)
    feats = np.concatenate([feats, theta_trend, jerk_local], axis=1)  # +1+3 = 24
    return feats.astype(np.float32)

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim, dim),
        )
        self.ln = nn.LayerNorm(dim)
    def forward(self, x): return self.ln(x + self.net(x))

class AccelField(nn.Module):
    def __init__(self, latent_dim=64, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 + 3 + latent_dim + 1, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
            ResBlock(hidden),
            nn.Linear(hidden, 3),
        )
    def forward(self, pos, vel, latent, speed):
        if speed.dim() == 1: speed = speed.unsqueeze(-1)
        return self.net(torch.cat([pos, vel, latent, speed], dim=-1))

class NeuralODE(nn.Module):
    def __init__(self, feat_dim=24, latent_dim=64, hidden=64, dt_pred=0.080):
        super().__init__()
        self.dt_pred = dt_pred
        self.backbone = nn.Sequential(
            nn.Linear(feat_dim, latent_dim),
            nn.LayerNorm(latent_dim), nn.GELU(),
            ResBlock(latent_dim), ResBlock(latent_dim),
        )
        self.accel = AccelField(latent_dim, hidden)
        self.damping = nn.Parameter(torch.tensor([0.1, 0.1, 0.1]))
        self.local_bias = nn.Parameter(torch.zeros(3))
        self.global_bias = nn.Parameter(torch.zeros(3))

    def _deriv(self, pos, vel, latent, speed):
        a = self.accel(pos, vel, latent, speed)
        return vel, -self.damping * vel + a

    def _rk4(self, pos, vel, latent, speed, dt):
        dp1, dv1 = self._deriv(pos, vel, latent, speed)
        dp2, dv2 = self._deriv(pos + 0.5*dt*dp1, vel + 0.5*dt*dv1, latent, speed)
        dp3, dv3 = self._deriv(pos + 0.5*dt*dp2, vel + 0.5*dt*dv2, latent, speed)
        dp4, dv4 = self._deriv(pos + dt*dp3,     vel + dt*dv3,     latent, speed)
        new_p = pos + (dt/6.0) * (dp1 + 2*dp2 + 2*dp3 + dp4)
        new_v = vel + (dt/6.0) * (dv1 + 2*dv2 + 2*dv3 + dv4)
        return new_p, new_v

    def forward(self, feats, init_vel, speed, p_last):
        latent = self.backbone(feats)
        pos = torch.zeros_like(init_vel)
        vel = init_vel
        pos, vel = self._rk4(pos, vel, latent, speed, self.dt_pred)
        local = pos + self.local_bias
        return p_last + local + self.global_bias

print('Neural ODE model defined.')


Neural ODE model defined.


## §7. Neural ODE 학습 (RETRAIN_ODE=True일 때만)

5 seed × 5 fold = 25회 학습, 총 약 10분.

`RETRAIN_ODE=False`면 드라이브의 `ode/ode_ens_test.npy` 사용.


In [9]:
# ============================================================
# §7. Neural ODE 학습 + 실행
# ============================================================
def train_ode_fold(tr_idx, va_idx, seed=42, epochs=19, batch=512, lr=4e-3, wd=1e-3):
    set_seed(seed)
    model = NeuralODE().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    feats_train = extract_ode_features(X_train)
    feats_tr = torch.tensor(feats_train[tr_idx], device=DEVICE)
    feats_va = torch.tensor(feats_train[va_idx], device=DEVICE)

    diffs = np.diff(X_train, axis=1)
    v_last = diffs[:, -1].astype(np.float32)
    speed = np.linalg.norm(v_last, axis=1).astype(np.float32)
    p_last = X_train[:, -1].astype(np.float32)

    v_tr = torch.tensor(v_last[tr_idx], device=DEVICE)
    v_va = torch.tensor(v_last[va_idx], device=DEVICE)
    sp_tr = torch.tensor(speed[tr_idx], device=DEVICE)
    sp_va = torch.tensor(speed[va_idx], device=DEVICE)
    pl_tr = torch.tensor(p_last[tr_idx], device=DEVICE)
    pl_va = torch.tensor(p_last[va_idx], device=DEVICE)
    y_tr  = torch.tensor(Y_train[tr_idx], dtype=torch.float32, device=DEVICE)
    y_va  = torch.tensor(Y_train[va_idx], dtype=torch.float32, device=DEVICE)

    n_tr = len(tr_idx)
    best_hr, best_state, best_pred = -1.0, None, None

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_tr, device=DEVICE)
        for i in range(0, n_tr, batch):
            idx = perm[i:i+batch]
            pred = model(feats_tr[idx], v_tr[idx], sp_tr[idx], pl_tr[idx])
            d = torch.norm(pred - y_tr[idx], dim=-1)
            loss = torch.sigmoid((d - R_HIT) / 0.002).mean()           # soft hit
            loss = loss + 126.309 * F.huber_loss(pred, y_tr[idx], delta=0.001026)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            pred_va = model(feats_va, v_va, sp_va, pl_va).cpu().numpy()
        hr = r_hit(pred_va, Y_train[va_idx])
        if hr > best_hr:
            best_hr = hr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_pred = pred_va
    return best_state, best_pred, best_hr

def predict_ode(model_state, X):
    feats = torch.tensor(extract_ode_features(X), device=DEVICE)
    diffs = np.diff(X, axis=1)
    v = torch.tensor(diffs[:, -1].astype(np.float32), device=DEVICE)
    s = torch.tensor(np.linalg.norm(diffs[:, -1], axis=1).astype(np.float32), device=DEVICE)
    p = torch.tensor(X[:, -1].astype(np.float32), device=DEVICE)
    model = NeuralODE().to(DEVICE)
    model.load_state_dict(model_state); model.eval()
    preds = []
    with torch.no_grad():
        bs = 2048
        for i in range(0, len(feats), bs):
            pr = model(feats[i:i+bs], v[i:i+bs], s[i:i+bs], p[i:i+bs])
            preds.append(pr.cpu().numpy())
    return np.concatenate(preds, axis=0)

if RETRAIN_ODE:
    print('=' * 60); print('Neural ODE training (5 seed x 5 fold)'); print('=' * 60)
    oof_ode = np.zeros_like(Y_train, dtype=np.float32)
    test_ode_seeds = []
    for seed in [42, 7, 77, 123, 2024]:
        print(f'\n--- Seed {seed} ---')
        oof_seed = np.zeros_like(Y_train, dtype=np.float32)
        test_seed_folds = []
        kf = KFold(n_splits=N_FOLD, shuffle=True, random_state=42)
        for fi, (tr, va) in enumerate(kf.split(np.arange(len(Y_train)))):
            t1 = time.time()
            state, pred_va, hr = train_ode_fold(tr, va, seed=seed, epochs=19)
            oof_seed[va] = pred_va
            test_pred = predict_ode(state, X_test)
            test_seed_folds.append(test_pred)
            print(f'  fold{fi}: hr={hr:.4f}  ({time.time()-t1:.0f}s)')
        oof_ode += oof_seed / 5.0
        test_ode_seeds.append(np.mean(test_seed_folds, axis=0))
    test_ode = np.mean(test_ode_seeds, axis=0)
    oof_ode_hr = r_hit(oof_ode, Y_train)
    print(f'\nNeural ODE OOF R-Hit: {oof_ode_hr:.4f}')
    np.save(OUT_DIR / 'ode_oof.npy', oof_ode)
    np.save(OUT_DIR / 'ode_test.npy', test_ode)
    print(f'Saved to {OUT_DIR}/ode_*.npy')
else:
    ode_dir = DRIVE_ASSETS / 'ode'
    assert (ode_dir / 'ode_ens_test.npy').exists(), f'Not found: {ode_dir}/ode_ens_test.npy'
    test_ode = np.load(ode_dir / 'ode_ens_test.npy')
    oof_ode  = np.load(ode_dir / 'ode_ens_oof.npy') if (ode_dir / 'ode_ens_oof.npy').exists() else None
    print(f'Neural ODE loaded: test={test_ode.shape}')
    if oof_ode is not None:
        print(f'  OOF R-Hit: {r_hit(oof_ode, Y_train):.4f}')


Neural ODE loaded: test=(10000, 3)
  OOF R-Hit: 0.6646


## §8. 블렌드 + Submission 생성

Phase C × Neural ODE 좌표 평균 (w=0.50).


In [10]:
# ============================================================
# §8. 블렌드 + submission
# ============================================================
W = 0.50  # ODE 가중치 (test 곡선 정점)
final_test = (1.0 - W) * test_c + W * test_ode

# OOF 점수 보고 (있을 때만)
if oof_c is not None and oof_ode is not None:
    print('=' * 60); print('OOF Validation'); print('=' * 60)
    print(f'  Phase C alone : {r_hit(oof_c, Y_train):.4f}')
    print(f'  Neural ODE    : {r_hit(oof_ode, Y_train):.4f}')
    print(f'  Blend (w={W})  : {r_hit((1-W)*oof_c + W*oof_ode, Y_train):.4f}')

# Submission 생성
sub = sample_sub.copy()
assert list(sub['id']) == test_ids, 'Submission id order mismatch!'
sub[['x','y','z']] = final_test
out_csv = OUT_DIR / 'submission.csv'
sub.to_csv(out_csv, index=False)

# Sanity check
arr = sub[['x','y','z']].values
print(f'\nSubmission saved: {out_csv}')
print(f'  shape: {sub.shape}')
print(f'  nan: {np.isnan(arr).any()}, inf: {np.isinf(arr).any()}')
print(f'  range: [{arr.min():.3f}, {arr.max():.3f}]')
print(sub.head())

# 백업
sub_c = sample_sub.copy(); sub_c[['x','y','z']] = test_c
sub_c.to_csv(OUT_DIR / 'submission_c_only.csv', index=False)
print(f'Backup (C only): {OUT_DIR}/submission_c_only.csv')


OOF Validation
  Phase C alone : 0.6606
  Neural ODE    : 0.6646
  Blend (w=0.5)  : 0.6671

Submission saved: /content/sub_out/submission.csv
  shape: (10000, 4)
  nan: False, inf: False
  range: [-2.595, 6.404]
           id         x         y         z
0  TEST_00001  3.989881 -1.052511  0.045355
1  TEST_00002  1.929678  0.617200  0.797529
2  TEST_00003  0.805865 -0.190620  0.147170
3  TEST_00004  3.527089 -1.645937 -0.941577
4  TEST_00005  2.302588  0.016714 -0.038972
Backup (C only): /content/sub_out/submission_c_only.csv


---

## 끝

`submission.csv`를 데이콘 제출 페이지에 업로드.

**Expected**: Public ≈ 0.6872 (블렌드), 0.684 (C only fallback).
